<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook places Phase 1 long-read samples in population-genetic space. It QC-filters and LD-prunes the GLnexus joint-call MatrixTable, computes PCA and relatedness, projects samples onto gnomAD and the 245k-sample short-read All of Us PCA, and supports selection of a diverse subset.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

Note: this notebook should run in a Hail VM environment.

# PCA of AoU-LR participants vs all AoU participants

In [ ]:
# standard imports
import os
import sys
import pandas as pd
import re

import math
import datetime
from dateutil import parser
from time import sleep
from typing import List
import pprint

from google.cloud import storage
my_storage_client = storage.Client()

from tqdm.notebook import tqdm
from tqdm import tqdm

In [ ]:
import firecloud.api as fapi

In [ ]:
import numpy as np

In [ ]:
import hail as hl
from hail.plot import show
from pprint import pprint

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import random

In [ ]:
# Load a Terra data table into a pandas DataFrame, with the entity ID in an entity:<type>_id column.
def load_table(namespace, workspace, root_entity_type):
    ent = fapi.get_entities(namespace, workspace, root_entity_type).json()
    tbl = pd.DataFrame(list(map(lambda e: e['attributes'], ent)))
    tbl[f"entity:{root_entity_type}_id"] = list(map(lambda f: f['name'], ent))

    return tbl

In [ ]:
# Account the notebook runs as.
current_account = fapi.whoami()

In [ ]:
current_account

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']       # the workspace bucket
workspace = os.environ['WORKSPACE_NAME']      # workspace name
namespace = os.environ['WORKSPACE_NAMESPACE'] # workspace namespace

In [ ]:
# The GLnexus joint-calling sample sets; take the 1,027-sample MatrixTable path and check its size.
tbl = load_table(namespace, workspace, 'ha-sample-grch38_set')

In [ ]:
tbl.head(4)

In [ ]:
joint_mt_path = list(tbl[tbl['entity:ha-sample-grch38_set_id'] == 'cohort_for_GLNexus_2023Q1_1027']['joint_mt'])[0]

In [ ]:
joint_mt_path

In [ ]:
!gsutil du -hcs {joint_mt_path}

In [ ]:
!gsutil ls -lh {joint_mt_path}

In [ ]:
# Start Hail, using GRCh38 coordinates.
hl.init(default_reference='GRCh38')

In [ ]:
joint_mt = hl.read_matrix_table(joint_mt_path)

In [ ]:
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
hl.summarize_variants(joint_mt)

In [ ]:
# Hail sample and variant QC metrics, then restrict to the autosomes.
# run sample QC and save into matrix table
joint_mt = hl.sample_qc(joint_mt)

In [ ]:
# run variant QC and save into matrix table
joint_mt = hl.variant_qc(joint_mt)

In [ ]:
test_intervals = [
    'chr1',  'chr2',  'chr3',  'chr4',  'chr5',  'chr6',
    'chr7',  'chr8',  'chr9',  'chr10', 'chr11', 'chr12',
    'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18',
    'chr19', 'chr20', 'chr21', 'chr22'
] 

In [ ]:
joint_mt = hl.filter_intervals(joint_mt, [hl.parse_locus_interval(x,) for x in test_intervals])
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
# filter for biallelic SNVs that are >95% call rate, >1% frequency, and remove sites far from HWE

In [ ]:
# Keep biallelic SNVs with call rate > 95%, alt allele frequency > 1%, and HWE p > 0.005, then LD-prune
# (r2 < 0.1 within 500 kb). The counts after each step are printed.
joint_mt = joint_mt.filter_rows(hl.len(joint_mt.alleles) == 2)
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
joint_mt = joint_mt.filter_rows(hl.is_snp(joint_mt.alleles[0], joint_mt.alleles[1]))
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.call_rate > 0.95)
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.AF[1] > 0.01)
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.p_value_hwe > 0.005)
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
pruned_variant_table = hl.ld_prune(joint_mt.GT, r2=0.1, bp_window_size=500000)
joint_mt = joint_mt.filter_rows(hl.is_defined(pruned_variant_table[joint_mt.row_key]))

In [ ]:
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
hl.summarize_variants(joint_mt)

In [ ]:
# Save the pruned site table and MatrixTable next to the joint-call MatrixTable.
pruned_tbl_path = f'{os.path.dirname(joint_mt_path)}/pruned_tbl.ht'
pruned_tbl_path

In [ ]:
pruned_mt_path = f'{os.path.dirname(joint_mt_path)}/pruned.mt'
pruned_mt_path

In [ ]:
pruned_variant_table.write(pruned_tbl_path, overwrite=False)

In [ ]:
joint_mt.write(pruned_mt_path, overwrite=False)

In [ ]:
pruned_mt = hl.read_matrix_table(pruned_mt_path)

In [ ]:
hl.summarize_variants(pruned_mt)

In [ ]:
# PCA on the pruned sites, then kinship with PC-Relate using the top PCs as covariates. A maximal
# independent set of the pairs with kinship > 0.1 picks the samples to remove so no related pair remains.
eigenvalues, pcs, loadings = hl.hwe_normalized_pca(pruned_mt.GT, k=10, compute_loadings=True)

In [ ]:
rel = hl.pc_relate(pruned_mt.GT,
                   0.01,
                   scores_expr=pcs[pruned_mt.col_key].scores,
                   min_kinship=0.1)

In [ ]:
related_samples_to_remove = hl.maximal_independent_set(rel.i, rel.j, keep=False)

In [ ]:
related_samples_to_remove.count()

In [ ]:
related_samples_path = f'{os.path.dirname(joint_mt_path)}/related_samples_to_remove.ht'
related_samples_path

related_samples_to_remove.write(related_samples_path)

In [ ]:
rel.show()

In [ ]:
pprint(eigenvalues)

In [ ]:
pcs.show(1027, width=100)

In [ ]:
# Plot PC1 vs PC2 of the long-read cohort alone.
pruned_mt = pruned_mt.annotate_cols(scores = pcs[pruned_mt.s].scores)

In [ ]:
pruned_mt.count()

In [ ]:
hl.plot.output_notebook()

p = hl.plot.scatter(pruned_mt.scores[0],
                    pruned_mt.scores[1],
                    hover_fields={'sample': pruned_mt.s},
                    title='PCA', xlabel='PC1', ylabel='PC2', collect_all=True)
show(p)

In [ ]:
# Project onto gnomAD v3.1's PCA and assign gnomAD population labels with gnomAD's random forest model.
!gsutil ls -lh gs://gcp-public-data--gnomad/release/3.1/pca/

In [ ]:
path_to_gnomad_loadings = 'gs://gcp-public-data--gnomad/release/3.1/pca/gnomad.v3.1.pca_loadings.ht'
path_to_gnomad_rf = 'gs://gcp-public-data--gnomad/release/3.1/pca/gnomad.v3.1.RF_fit.pkl'

In [ ]:
import pickle
import hail as hl
from gnomad.sample_qc.ancestry import assign_population_pcs

In [ ]:
# Uses joint_mt (the QC-filtered, LD-pruned sites) for the projection.
# Load MatrixTable for projection and gnomAD loadings Hail Table
#mt_to_project = hl.read_matrix_table(path_to_matrix_table)
mt_to_project = joint_mt
loadings_ht = hl.read_table(path_to_gnomad_loadings)

In [ ]:
# Project new genotypes onto loadings
ht = hl.experimental.pc_project(
    mt_to_project.GT,
    loadings_ht.loadings,
    loadings_ht.pca_af,
)

In [ ]:
# Assign global ancestry using the gnomAD RF model and PC project scores
# Loading of the v2 RF model requires an older version of scikit-learn, this can be installed using pip install -U scikit-learn==0.21.3
with hl.hadoop_open(path_to_gnomad_rf, "rb") as f:
    fit = pickle.load(f)

# Reduce the scores to only those used in the RF model, this was 6 for v2 and 16 for v3.1
num_pcs = fit.n_features_
ht = ht.annotate(scores=ht.scores[:num_pcs])
ht, rf_model = assign_population_pcs(
    ht,
    pc_cols=ht.scores,
    fit=fit,
)

# The returned Hail Table includes the imputed population labels and RF probabilities for each gnomAD global population

In [ ]:
ht.show(1030)

In [ ]:
# Per-sample probability of each gnomAD population, sorted by AFR probability, as stacked bars
# (all samples, then the first 160).
q = ht.collect()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
samples = np.array([])
prob_afr = np.array([])
prob_ami = np.array([])
prob_amr = np.array([])
prob_asj = np.array([])
prob_eas = np.array([])
prob_fin = np.array([])
prob_mid = np.array([])
prob_nfe = np.array([])
prob_oth = np.array([])
prob_sas = np.array([])

for i in range(0, len(q)):
    samples = np.append(samples, q[i].s)
    prob_afr = np.append(prob_afr, q[i].prob_afr)
    prob_ami = np.append(prob_ami, q[i].prob_ami)
    prob_amr = np.append(prob_amr, q[i].prob_amr)
    prob_asj = np.append(prob_asj, q[i].prob_asj)
    prob_eas = np.append(prob_eas, q[i].prob_eas)
    prob_fin = np.append(prob_fin, q[i].prob_fin)
    prob_mid = np.append(prob_mid, q[i].prob_mid)
    prob_nfe = np.append(prob_nfe, q[i].prob_nfe)
    prob_oth = np.append(prob_oth, q[i].prob_oth)
    prob_sas = np.append(prob_sas, q[i].prob_sas)
    
p = prob_afr.argsort()

samples = samples[p]
prob_afr = prob_afr[p]
prob_ami = prob_ami[p]
prob_amr = prob_amr[p]
prob_asj = prob_asj[p]
prob_eas = prob_eas[p]
prob_fin = prob_fin[p]
prob_mid = prob_mid[p]
prob_nfe = prob_nfe[p]
prob_oth = prob_oth[p]
prob_sas = prob_sas[p]

In [ ]:
plt.rcParams.update({'font.size': 15})


In [ ]:
fig = plt.figure(figsize=(20,8))
ax = fig.add_subplot(111)

plt.bar(samples, prob_afr, color='#7F1391', align='edge', width=1.0)
plt.bar(samples, prob_ami, color='#FEBFCB', align='edge', width=1.0, bottom=prob_afr)
plt.bar(samples, prob_amr, color='#EE2324', align='edge', width=1.0, bottom=prob_afr + prob_ami)
plt.bar(samples, prob_asj, color='#FB683E', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr)
plt.bar(samples, prob_eas, color='#127D31', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj)
plt.bar(samples, prob_fin, color='#072f69', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas)
plt.bar(samples, prob_mid, color='#36D335', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin)
plt.bar(samples, prob_nfe, color='#67A8D2', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid)
plt.bar(samples, prob_oth, color='#7F7F7F', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid + prob_nfe)
plt.bar(samples, prob_sas, color='#FE9911', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid + prob_nfe + prob_oth)

plt.xlabel('Participant')
plt.ylabel('Population probability')

frame1 = plt.gca()
frame1.axes.get_xaxis().set_ticks([])

plt.show()

In [ ]:
fig = plt.figure(figsize=(20,8))
ax = fig.add_subplot(111)

plt.bar(samples, prob_afr, color='#7F1391', align='edge', width=1.0)
plt.bar(samples, prob_ami, color='#FEBFCB', align='edge', width=1.0, bottom=prob_afr)
plt.bar(samples, prob_amr, color='#EE2324', align='edge', width=1.0, bottom=prob_afr + prob_ami)
plt.bar(samples, prob_asj, color='#FB683E', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr)
plt.bar(samples, prob_eas, color='#127D31', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj)
plt.bar(samples, prob_fin, color='#072f69', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas)
plt.bar(samples, prob_mid, color='#36D335', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin)
plt.bar(samples, prob_nfe, color='#67A8D2', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid)
plt.bar(samples, prob_oth, color='#7F7F7F', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid + prob_nfe)
plt.bar(samples, prob_sas, color='#FE9911', align='edge', width=1.0, bottom=prob_afr + prob_ami + prob_amr + prob_asj + prob_eas + prob_fin + prob_mid + prob_nfe + prob_oth)

plt.xlabel('Participant')
plt.ylabel('Population probability')
plt.xlim(0, 160)

frame1 = plt.gca()
frame1.axes.get_xaxis().set_ticks([])

plt.show()

In [ ]:
# Projected PC1 vs PC2.
hl.plot.output_notebook()

p = hl.plot.scatter(ht.pca_scores[0],
                    ht.pca_scores[1],
                    hover_fields={'sample': ht.s},
                    title='PCA', xlabel='PC1', ylabel='PC2', collect_all=True)
show(p)

### Project long read SNVs/small indels into PCA space of the 245k short read samples.

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# The AoU short-read ancestry pipeline's PCA loadings and allele frequencies (about 245k samples).
!gsutil ls -lh {AOU_LR_DATA}/aou-wgs-delta-aux_gq0/ancestry

In [ ]:
loadings_ht = hl.methods.read_table(f'{AOU_LR_DATA}/aou-wgs-delta-aux_gq0/ancestry/delta_v1_gt_no_ext_aian_gq0_prod_loadings.ht')

In [ ]:
loadings_ht.describe()

In [ ]:
# Project the long-read samples into that PCA space and save the scores.
testing_ht = hl.experimental.pc_project(joint_mt.GT, loadings_ht.loadings, loadings_ht.af)

In [ ]:
!gsutil ls -lh {AOU_LR_DATA}/outputs/GRCh38/

In [ ]:
testing_ht_path = f'{AOU_LR_DATA}/outputs/GRCh38/scratch/pca_project/testing.ht'

In [ ]:
testing_ht.write(testing_ht_path, overwrite=False)

In [ ]:
testing_ht = hl.read_table(testing_ht_path)

In [ ]:
testing_ht.count()

In [ ]:
testing_ht.show(5)

In [ ]:
testing_ht.describe()

In [ ]:
!gsutil ls -lh {AOU_LR_DATA}/aou-wgs-delta-aux_gq0/*/

In [ ]:
# The short-read pipeline's ancestry predictions and PC scores for every short-read participant.
!gsutil cp {AOU_LR_DATA}/aou-wgs-delta-aux_gq0/ancestry/delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.tsv .

In [ ]:
ancestry_preds = pd.read_table('delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.tsv')

In [ ]:
ancestry_preds['research_id'] = ancestry_preds['research_id'].astype(str)

In [ ]:
ancestry_preds

In [ ]:
# Participants per predicted ancestry, and the predictions for the long-read participants.
from collections import Counter

Counter(list(ancestry_preds['ancestry_pred_other']))

In [ ]:
ancestry_preds[ancestry_preds['ancestry_pred_other'] == 'afr']

In [ ]:
ancestry_preds[ancestry_preds['research_id'].isin(testing_ht.s.collect())]

In [ ]:
# The same predictions as a Hail table.
ancestry_preds_tbl = hl.import_table(
    f'{AOU_LR_DATA}/aou-wgs-delta-aux_gq0/ancestry/delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.tsv',
    types={ 'pca_features': hl.tarray(hl.tfloat64) }
)

ancestry_preds_tbl.show(20)

In [ ]:
ancestry_preds_tbl.count()

In [ ]:
l = ancestry_preds_tbl.ancestry_pred_other.collect()

In [ ]:
Counter(l)

In [ ]:
# Colours for each continental ancestry group.
color_map = {
    'afr': '#D573BE',
    'amr': '#519D3E',
    'eas': '#C53A32',
    'eur': '#3976AF',
    'mid': '#EF8536',
    'sas': '#866BE8',
    'oth': '#84584E'
}

In [ ]:
# Short-read participants on PC1 vs PC2, coloured by predicted ancestry.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color=[f'{color_map[item]}1A' for item in l],
    s=2
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
# Overlay the long-read samples' projected scores.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color=[f'{color_map[item]}1A' for item in l],
    s=2
)

plt.scatter(
    testing_ht.scores[0].collect(),
    testing_ht.scores[1].collect(),
    marker="^",
    edgecolor="#10454FBB",
    color="#ffffff11",
    s=10
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
# Predicted ancestry counts among the long-read participants, and their PC scores from the short-read
# pipeline (a1), for comparison with the scores projected from long-read genotypes.
q = ancestry_preds[ancestry_preds['research_id'].isin(testing_ht.s.collect())]

In [ ]:
Counter(q['ancestry_pred_other'])

In [ ]:
a1 = [[float(f) for f in re.split(",", re.sub(r"[\[\]]", "", item))] for item in list(q.pca_features)]

In [ ]:
# Short-read-derived scores (green) vs long-read projected scores (blue) for the same participants.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color=[f'{color_map[item]}1A' for item in l],
    s=2
)

plt.scatter(
    [item[0] for item in a1],
    [item[1] for item in a1],
    marker="^",
    edgecolor="#BDE038BB",
    color="#ffffff11",
    s=10
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color=[f'{color_map[item]}1A' for item in l],
    s=2
)

plt.scatter(
    [item[0] for item in a1],
    [item[1] for item in a1],
    marker="^",
    edgecolor="#BDE038BB",
    color="#ffffff11",
    s=10
)

plt.scatter(
    testing_ht.scores[0].collect(),
    testing_ht.scores[1].collect(),
    marker="^",
    edgecolor="#10454FBB",
    color="#ffffff11",
    s=5
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color='#ffffff',
    s=2
)

plt.scatter(
    [item[0] for item in a1],
    [item[1] for item in a1],
    marker="^",
    edgecolor="#BDE038BB",
    color="#ffffff11",
    s=10
)

plt.scatter(
    testing_ht.scores[0].collect(),
    testing_ht.scores[1].collect(),
    marker="^",
    edgecolor="#10454FBB",
    color="#ffffff11",
    s=5
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
# Difference between the two sets of scores per participant. This assumes a1 and testing_ht list the
# participants in the same order.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    np.subtract([item[0] for item in a1], testing_ht.scores[0].collect()),
    np.subtract([item[1] for item in a1], testing_ht.scores[1].collect()),
    marker="^",
    edgecolor="#10454FBB",
    color="#ffffff11",
    s=5
)

plt.xlabel("diff PC1")
plt.ylabel("diff PC2")

plt.show()

In [ ]:
len(testing_ht.scores[0].collect())

In [ ]:
len(testing_ht.scores[1].collect())

In [ ]:
# Pick about 55 samples, from the outlying parts of the plot plus a random fill, and highlight them in red.
random.seed(15)

b1 = testing_ht.scores[0].collect()
b2 = testing_ht.scores[1].collect()

vi0 = list(filter(lambda i: b1[i] > -0.1 and b2[i] < 0.03, range(len(b2))))
vi1 = list(filter(lambda i: b2[i] < -0.02, range(len(b2)))) + list(filter(lambda i: b1[i] > 0.08, range(len(b1))))
vi2 = random.sample(range(0, 1027), 55 - len(set(vi0+vi1)) + 1)
vi = list(set(vi0+vi1+vi2))

c1 = [b1[index] for index in vi]
c2 = [b2[index] for index in vi]

s_all = list(testing_ht.s.collect())
s = [s_all[index] for index in vi]

plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

plt.scatter(
    ancestry_preds_tbl.pca_features[0].collect(),
    ancestry_preds_tbl.pca_features[1].collect(),
    color='#ffffff',
    s=2
)

plt.scatter(
    [item[0] for item in a1],
    [item[1] for item in a1],
    marker="^",
    edgecolor="#BDE038BB",
    color="#ffffff11",
    s=10
)

plt.scatter(
    testing_ht.scores[0].collect(),
    testing_ht.scores[1].collect(),
    marker="^",
    edgecolor="#10454FBB",
    color="#ffffff11",
    s=5
)

plt.scatter(
    c1,
    c2,
    marker="^",
    edgecolor="#FF0000BB",
    color="#ffffff11",
    s=5
)


plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

In [ ]:
s

In [ ]:
print(len(vi))

print(len(c1))
print(len(c2))

print(len(s))

In [ ]:
len(set(vi0+vi1))

In [ ]:
list(filter(lambda i: b2[i] < -0.02, range(len(b2))))

In [ ]:
list(filter(lambda i: b1[i] > 0.08, range(len(b1))))

In [ ]:
s

In [ ]:
vi

In [ ]:
vi2

In [ ]:
c1

In [ ]:
list(filter(lambda i: b1[i] > 0.08, range(len(b1))))

In [ ]:
len(s)

In [ ]:
s